# 9. パラメータ最適化：グリッド検索による取引ルール評価

## 概要
既存テクニカル指標（SMA/MACD/RSI）の固定パラメータをグリッド検索で最適化します。
複数銘柄でバックテストを実施し、ウォークフォワード分析で過学習チェックを行います。

## リサーチ目標
1. **SMA クロス戦略**の短期/長期日数を最適化
2. **MACD 戦略**の fast/slow/signal パラメータを最適化  
3. **RSI クロス戦略**の短期/長期日数を最適化
4. **複数銘柄での汎化性能**を検証
5. **ウォークフォワード分析**で過学習を可視化

## 1. ライブラリと実行環境の設定

### このセルの説明
- 何をしているか？
  - 分析に必要なライブラリ（データ処理、テクニカル指標、バックテスト、可視化）を読み込み、乱数シードや描画設定を初期化しています。
- 何のためにしているか？
  - 後続セルで同一条件の再現実験を行うために、実行環境を固定します。とくに `Backtest` クラスと `Strategy` クラスを使うための前準備です。
- 理論的背景とモデル
  - 乱数シード固定は、推定量 $\hat{\theta}$ の再現性を確保するための基本操作です。
  - 同一データ $D$ と同一設定 $s$ に対して、出力が
  $$
  f(D, s) = \text{constant}
  $$
  となる状態を目指します。

In [29]:
import os
import sys
import warnings
import requests
from pathlib import Path
from dotenv import load_dotenv
import datetime as dt
import itertools

# データ処理・統計
import pandas as pd
import numpy as np
from functools import partial

# テクニカル分析
import talib as ta
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# 可視化
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# 設定
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

# APIベースURL
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. データ読み込みと前処理パイプライン

trn_white_list に登録された銘柄をローカル API から取得し、株価の時系列データをバックテスト向けに整形します。

### このセルの説明
- 何をしているか？
  - ローカル API の `stocks` エンドポイントへ GET リクエストを送り、`trn_white_list` 登録銘柄からテスト対象を選定しています。
- 何のためにしているか？
  - 最適化結果の対象集合 $\mathcal{U}$（ユニバース）を明確化し、再現可能な評価対象を定義するためです。
- 理論的背景とモデル
  - 銘柄集合を
  $$
  \mathcal{U}=\{(c_i,m_i,n_i)\}_{i=1}^{N}
  $$
  と置くと、以降の評価は $\mathcal{U}$ 上で定義されます。
  - 対象が変わると推定された最適パラメータ $\theta^*$ も変化しうるため、ユニバース固定が重要です。

In [ ]:
# API から trn_white_list テーブルの銘柄一覧を取得
try:
    stocks_url = f"{API_BASE_URL}/api/v1/stocks"
    stocks_response = requests.get(stocks_url, timeout=60)
    stocks_response.raise_for_status()

    all_stocks = stocks_response.json().get("results", [])

    # 最初の4銘柄をサンプルとして選択
    TARGET_STOCKS = []
    for stock in all_stocks[:4]:
        code = stock.get("code")
        market = stock.get("market")
        name = stock.get("name") or code
        currency = stock.get("currency") or 'JPY'  # APIレスポンスのcurrencyを優先
        TARGET_STOCKS.append({
            "code": code,
            "market": market,
            "name": name,
            "currency": currency,
        })

    print(f"✓ API から {len(all_stocks)} 銘柄を取得しました")
    print(f"✓ テスト対象銘柄: {len(TARGET_STOCKS)} 銘柄")
except Exception as e:
    print(f"✗ 銘柄リスト取得エラー: {e}")
    TARGET_STOCKS = []

# データ取得期間
DATA_START = dt.datetime(2023, 1, 1)
DATA_END = dt.datetime(2024, 12, 31)

# バックテスト設定
CASH = 100000  # 初期資金（円）
COMMISSION = 0.001  # 片道手数料 0.1%

print(f"取得期間: {DATA_START.date()} ～ {DATA_END.date()}")
print("テスト対象銘柄:")
for stock in TARGET_STOCKS:
    print(f"  - {stock['name']} ({stock['code']}) [{stock['market']}] {stock['currency']}")

✓ API から 191 銘柄を取得しました
✓ テスト対象銘柄: 4 銘柄
取得期間: 2023-01-01 ～ 2024-12-31
テスト対象銘柄:
  - GD (GD) [NYSE] USD
  - GDX (GDX) [NYSE] USD
  - GDXJ (GDXJ) [NYSE] USD
  - GEV (GEV) [NYSE] USD


### このセルの説明
- 何をしているか？
  - API から取得した時系列データを `Backtesting.py` が要求する OHLCV 形式に正規化し、`fetch_stock_data` で取得と前処理を関数化しています。
  - あわせて為替換算のために、Frankfurter API から対象通貨→JPY レートを取得する関数を定義しています。
- 何のためにしているか？
  - `Backtest` の入力仕様（`Open, High, Low, Close, Volume`）に合わせ、欠損・型不一致による推定バイアスや実行エラーを防ぐためです。
  - 売買日の為替で円建て損益を計算するためです。
- 参照している為替API
  - Frankfurter API: https://frankfurter.dev/
- 理論的背景とモデル
  - 日次収益率は
  $$
  r_t = \frac{P_t}{P_{t-1}} - 1
  $$
  で定義され、$P_t$ は通常 `Close` 価格です。
  - 前処理は写像
  $$
  T: D_{raw} \rightarrow D_{bt}
  $$
  とみなせ、最適化の前提条件を満たす $D_{bt}$ を構成します。

In [31]:
# 為替レート取得のキャッシュ
FX_RATE_CACHE = {}


def prepare_backtest_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    API レスポンスを Backtesting.py 互換の OHLCV 形式へ整形する。

    Backtesting.py は 'Open', 'High', 'Low', 'Close', 'Volume' 列を要求するため、
    API が返す小文字列名をここで正規化する。
    """
    required_columns = ['open', 'high', 'low', 'close']
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"不足している列があります: {missing_columns}")

    rename_map = {
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume',
    }

    normalized_df = df.rename(columns=rename_map).copy()

    if 'Volume' not in normalized_df.columns:
        normalized_df['Volume'] = 0

    numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    for column in numeric_columns:
        normalized_df[column] = pd.to_numeric(normalized_df[column], errors='coerce')

    normalized_df = normalized_df[numeric_columns].dropna(subset=['Open', 'High', 'Low', 'Close'])
    normalized_df = normalized_df.sort_index()

    if normalized_df.empty:
        raise ValueError('バックテスト用データが空です')

    return normalized_df


def fetch_fx_rate_to_jpy(currency: str, trade_date: dt.date) -> float:
    """
    約定日の対象通貨->JPY レートを取得する。

    参照API: Frankfurter API (https://frankfurter.dev/)
    休場日を考慮し、最大7営業日前まで遡ってレートを探索する。
    """
    if currency == 'JPY':
        return 1.0

    cache_key = (currency, trade_date.isoformat())
    if cache_key in FX_RATE_CACHE:
        return FX_RATE_CACHE[cache_key]

    for lag in range(0, 8):
        query_date = trade_date - dt.timedelta(days=lag)
        url = f"https://api.frankfurter.app/{query_date.isoformat()}"
        params = {'from': currency, 'to': 'JPY'}

        try:
            response = requests.get(url, params=params, timeout=20)
            response.raise_for_status()
            rates = response.json().get('rates', {})
            jpy_rate = rates.get('JPY')
            if jpy_rate is not None:
                FX_RATE_CACHE[cache_key] = float(jpy_rate)
                return float(jpy_rate)
        except requests.RequestException:
            continue

    raise ValueError(f"為替レート取得失敗: {currency} -> JPY ({trade_date})")


def calc_jpy_trade_amounts(stats, currency: str) -> dict:
    """
    取引明細を約定日の為替で円換算し、円建て買付・売却・損益を返す。
    """
    trades_df = getattr(stats, '_trades', pd.DataFrame())
    if trades_df is None or trades_df.empty:
        return {
            'buy_jpy': 0.0,
            'sell_jpy': 0.0,
            'pnl_jpy': 0.0,
            'return_jpy_pct': np.nan,
            'trade_count': 0,
        }

    total_buy_jpy = 0.0
    total_sell_jpy = 0.0

    for _, trade in trades_df.iterrows():
        qty = abs(float(trade['Size']))
        entry_price = float(trade['EntryPrice'])
        exit_price = float(trade['ExitPrice'])

        entry_date = pd.to_datetime(trade['EntryTime']).date()
        exit_date = pd.to_datetime(trade['ExitTime']).date()

        entry_fx = fetch_fx_rate_to_jpy(currency, entry_date)
        exit_fx = fetch_fx_rate_to_jpy(currency, exit_date)

        buy_jpy = entry_price * qty * entry_fx
        sell_jpy = exit_price * qty * exit_fx

        total_buy_jpy += buy_jpy
        total_sell_jpy += sell_jpy

    pnl_jpy = total_sell_jpy - total_buy_jpy
    return_jpy_pct = (pnl_jpy / total_buy_jpy * 100.0) if total_buy_jpy > 0 else np.nan

    return {
        'buy_jpy': total_buy_jpy,
        'sell_jpy': total_sell_jpy,
        'pnl_jpy': pnl_jpy,
        'return_jpy_pct': return_jpy_pct,
        'trade_count': len(trades_df),
    }


def fetch_stock_data(code: str, market: str, start_date: dt.datetime, end_date: dt.datetime) -> pd.DataFrame:
    """
    API の /api/v1/time_series_data エンドポイントから株価データを取得し、
    Backtesting.py で使用できる DataFrame に整形する。
    """
    get_url = f"{API_BASE_URL}/api/v1/time_series_data"
    get_params = {
        "code": code,
        "market": market,
        "start": start_date.date().isoformat(),
        "end": end_date.date().isoformat(),
    }

    response = requests.get(get_url, params=get_params, timeout=60)
    response.raise_for_status()

    payload = response.json()
    rows = payload.get("results", [])
    if not rows:
        raise ValueError(f"No data returned for {code}")

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()

    normalized_df = prepare_backtest_data(df)
    return normalized_df


# テスト：最初の銘柄を取得
if TARGET_STOCKS:
    sample_code = TARGET_STOCKS[0]["code"]
    sample_market = TARGET_STOCKS[0]["market"]
    sample_name = TARGET_STOCKS[0]["name"]
    sample_currency = TARGET_STOCKS[0]["currency"]

    sample_df = fetch_stock_data(sample_code, sample_market, DATA_START, DATA_END)

    print(f"\n✓ Sample data loaded: {sample_name} ({sample_code})")
    print(f"  Currency: {sample_currency}")
    print(f"  Shape: {sample_df.shape}")
    print(f"  Date range: {sample_df.index[0].date()} ～ {sample_df.index[-1].date()}")
    print(f"  Columns: {sample_df.columns.tolist()}")
    print(f"\n{sample_df.head(3)}")
else:
    print("✗ No target stocks available")


✓ Sample data loaded: GD (GD)
  Currency: USD
  Shape: (502, 5)
  Date range: 2023-01-03 ～ 2024-12-31
  Columns: ['Open', 'High', 'Low', 'Close', 'Volume']

                  Open        High         Low       Close   Volume
date                                                               
2023-01-03  231.250122  232.003590  229.129245  230.552465   777900
2023-01-04  228.105988  231.398923  226.003719  230.217567  1076700
2023-01-05  227.547867  229.268760  226.775791  227.482760   945800


## 3. 既存ルールの関数化（ベースライン実装）

取引戦略を `backtesting.Strategy` として実装します。

### このセルの説明
- 何をしているか？
  - SMA クロス、MACD クロス、RSI クロスの3戦略を `Strategy` クラスとして実装しています。
- 何のためにしているか？
  - 各戦略のパラメータ（例: `n1`, `n2`, `fast`, `slow`, `signal_period`）を最適化対象変数として扱えるようにするためです。
- 理論的背景とモデル
  - SMA:
  $$
  SMA_t(N)=\frac{1}{N}\sum_{i=0}^{N-1}P_{t-i}
  $$
  - MACD:
  $$
  MACD_t=EMA_t(\text{fast})-EMA_t(\text{slow}),\quad Signal_t=EMA_t(MACD,\text{signal})
  $$
  - RSI:
  $$
  RSI_t=100-\frac{100}{1+RS_t}
  $$
  交差条件を売買シグナル $s_t\in\{-1,0,1\}$ に写像して売買判断を行います。

In [32]:
class SMACrossStrategy(Strategy):
    """
    SMA クロス戦略：短期MA > 長期MA の時に買い、短期MA < 長期MA の時に売却
    
    パラメータ:
        - n1 (int): 短期 SMA の日数（デフォルト: 5）
        - n2 (int): 長期 SMA の日数（デフォルト: 25）
    """
    n1 = 5
    n2 = 25
    
    def init(self):
        # 移動平均線を計算
        close = self.data.Close
        self.ma1 = self.I(ta.SMA, close, self.n1)
        self.ma2 = self.I(ta.SMA, close, self.n2)
    
    def next(self):
        # クロスオーバーシグナル
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            if self.position:
                self.position.close()


class MACDCrossStrategy(Strategy):
    """
    MACD クロス戦略：MACD > Signal の時に買い、MACD < Signal の時に売却
    
    パラメータ:
        - fast (int): MACD fast 期間（デフォルト: 12）
        - slow (int): MACD slow 期間（デフォルト: 26）
        - signal (int): Signal 期間（デフォルト: 9）
    """
    fast = 12
    slow = 26
    signal_period = 9
    
    def init(self):
        # MACD を計算
        close = self.data.Close
        macd, macd_signal, _ = self.I(
            ta.MACD, close, 
            fastperiod=self.fast, 
            slowperiod=self.slow, 
            signalperiod=self.signal_period
        )
        self.macd = macd
        self.macd_signal = macd_signal
    
    def next(self):
        if crossover(self.macd, self.macd_signal):
            self.buy()
        elif crossover(self.macd_signal, self.macd):
            if self.position:
                self.position.close()


class RSICrossStrategy(Strategy):
    """
    RSI クロス戦略：短期RSI > 長期RSI の時に買い、短期RSI < 長期RSI の時に売却
    
    パラメータ:
        - n1 (int): 短期 RSI の日数（デフォルト: 14）
        - n2 (int): 長期 RSI の日数（デフォルト: 28）
    """
    n1 = 14
    n2 = 28
    
    def init(self):
        # RSI を計算
        close = self.data.Close
        self.rsi1 = self.I(ta.RSI, close, self.n1)
        self.rsi2 = self.I(ta.RSI, close, self.n2)
    
    def next(self):
        if crossover(self.rsi1, self.rsi2):
            self.buy()
        elif crossover(self.rsi2, self.rsi1):
            if self.position:
                self.position.close()

print("✓ Strategy classes defined: SMA, MACD, RSI")

✓ Strategy classes defined: SMA, MACD, RSI


### このセルの説明
- 何をしているか？
  - デフォルトパラメータ（SMA 5/25）で単一バックテストを実行し、基準性能を出しています。
- 何のためにしているか？
  - 最適化後性能との比較基準（ベースライン）を作るためです。
- 理論的背景とモデル
  - 評価対象は代表的に累積収益率
  $$
  R = \frac{V_T - V_0}{V_0}\times 100
  $$
  と Sharpe 比率
  $$
  S = \frac{\mathbb{E}[r_p-r_f]}{\sigma(r_p-r_f)}
  $$
  です。ここで $V_t$ は資産価値、$r_p$ は戦略収益率です。

In [33]:
# デフォルトパラメータでベースラインテスト
print("\n" + "=" * 60)
print("ベースラインテスト（デフォルトパラメータ）")
print("=" * 60)

bt = Backtest(sample_df, SMACrossStrategy, cash=CASH, commission=COMMISSION)
baseline_result = bt.run()

print("\nSMA クロス戦略（5/25 日）の結果:")
print(f"  最終資産: ¥{baseline_result['Equity Final [$]']:,.0f}")
print(f"  総リターン: {baseline_result['Return [%]']:.2f}%")
print(f"  Sharpe 比率: {baseline_result['Sharpe Ratio']:.2f}")
print(f"  最大ドローダウン: {baseline_result['Max. Drawdown [%]']:.2f}%")
print(f"  トレード数: {int(baseline_result['# Trades'])}")
print(f"  勝率: {baseline_result['Win Rate [%]']:.2f}%")


ベースラインテスト（デフォルトパラメータ）


Backtest.run:   0%|          | 0/477 [00:00<?, ?bar/s]


SMA クロス戦略（5/25 日）の結果:
  最終資産: ¥94,030
  総リターン: -5.97%
  Sharpe 比率: -0.23
  最大ドローダウン: -18.08%
  トレード数: 12
  勝率: 41.67%


## 4. グリッド検索による SMA パラメータ最適化

複数の短期/長期 MA 組み合わせを試行し、最良パラメータを探索します。

### このセルの説明
- 何をしているか？
  - SMA の短期・長期窓を全組合せで探索し、各組合せの収益率や Sharpe 比率を記録しています。
- 何のためにしているか？
  - `SMAOptStrategy.n1` と `SMAOptStrategy.n2` を最適化し、目的指標を最大化するためです。
- 理論的背景とモデル
  - パラメータ最適化は
  $$
  \theta^* = \arg\max_{\theta\in\Theta} J(\theta)
  $$
  で表せます。ここで $\theta=(n_1,n_2)$、$J$ は Sharpe 比率などの評価関数です。
  - 制約条件は $n_1<n_2$ として探索空間を定義します。

In [34]:
def grid_search_sma(df: pd.DataFrame, 
                    short_ma_range: list, 
                    long_ma_range: list,
                    cash: float = CASH,
                    commission: float = COMMISSION) -> pd.DataFrame:
    """
    SMA クロス戦略のグリッド検索を実行
    
    Args:
        df (pd.DataFrame): OHLCV データ
        short_ma_range (list): 短期 MA 日数のリスト
        long_ma_range (list): 長期 MA 日数のリスト
        cash (float): 初期資金
        commission (float): 手数料
    
    Returns:
        pd.DataFrame: 全試行の結果（パラメータ、評価指標）
    """
    results = []

    valid_combinations = [(n1, n2) for n1, n2 in itertools.product(short_ma_range, long_ma_range) if n1 < n2]
    total_combinations = len(valid_combinations)

    for current, (n1, n2) in enumerate(valid_combinations, start=1):
        class SMAOptStrategy(SMACrossStrategy):
            pass

        SMAOptStrategy.n1 = n1
        SMAOptStrategy.n2 = n2

        bt = Backtest(df, SMAOptStrategy, cash=cash, commission=commission)
        result = bt.run()

        results.append({
            'n1': n1,
            'n2': n2,
            'Return [%]': result['Return [%]'],
            'Sharpe Ratio': result['Sharpe Ratio'],
            'Max. Drawdown [%]': result['Max. Drawdown [%]'],
            'Win Rate [%]': result['Win Rate [%]'],
            '# Trades': result['# Trades'],
            'Equity Final [$]': result['Equity Final [$]'],
        })

        if current % 5 == 0 or current == total_combinations:
            print(f"  進捗: {current}/{total_combinations}")

    return pd.DataFrame(results)

# グリッド検索パラメータ
SHORT_MA_RANGE = [3, 5, 7, 10]
LONG_MA_RANGE = [15, 20, 25, 30, 50]

print("\nSMA グリッド検索を実行中...")
print(f"短期 MA: {SHORT_MA_RANGE}")
print(f"長期 MA: {LONG_MA_RANGE}")

sma_results = grid_search_sma(sample_df, SHORT_MA_RANGE, LONG_MA_RANGE)

print(f"\n✓ グリッド検索完了: {len(sma_results)} 組合せを試行")
print("\nTop 5 結果（Sharpe 比率で評価）:")
print(sma_results.nlargest(5, 'Sharpe Ratio')[['n1', 'n2', 'Return [%]', 'Sharpe Ratio', '# Trades']])


SMA グリッド検索を実行中...
短期 MA: [3, 5, 7, 10]
長期 MA: [15, 20, 25, 30, 50]


Backtest.run:   0%|          | 0/487 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/482 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/477 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/472 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/482 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/477 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/472 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/452 [00:00<?, ?bar/s]

  進捗: 20/20

✓ グリッド検索完了: 20 組合せを試行

Top 5 結果（Sharpe 比率で評価）:
    n1  n2  Return [%]  Sharpe Ratio  # Trades
14   7  50   16.714227      0.543094         7
19  10  50   15.540109      0.520665         5
6    5  20    6.329744      0.198670        17
16  10  20    4.517808      0.176499        13
9    5  50    3.068359      0.112998        10


## 5. MACD パラメータ最適化

### このセルの説明
- 何をしているか？
  - MACD の `fast`, `slow`, `signal` の3変数をグリッド探索し、性能指標を比較しています。
- 何のためにしているか？
  - `ta.MACD` に与える引数（`fastperiod`, `slowperiod`, `signalperiod`）を最適化するためです。
- 理論的背景とモデル
  - MACD は
  $$
  MACD_t = EMA_t(fast)-EMA_t(slow)
  $$
  $$
  Hist_t = MACD_t-EMA_t(MACD,signal)
  $$
  で定義されます。
  - これはトレンドの差分（短期と長期の乖離）を利用したモデルで、交差点をシグナル化します。

In [35]:
def grid_search_macd(df: pd.DataFrame,
                     fast_range: list,
                     slow_range: list,
                     signal_range: list,
                     cash: float = CASH,
                     commission: float = COMMISSION) -> pd.DataFrame:
    """
    MACD クロス戦略のグリッド検索
    """
    results = []
    
    total_combinations = len(fast_range) * len(slow_range) * len(signal_range)
    current = 0
    
    for fast, slow, signal in itertools.product(fast_range, slow_range, signal_range):
        if fast >= slow:  # fast < slow の条件をチェック
            continue
        
        current += 1
        
        class MACDOptStrategy(MACDCrossStrategy):
            pass
        
        MACDOptStrategy.fast = fast
        MACDOptStrategy.slow = slow
        MACDOptStrategy.signal_period = signal
        
        bt = Backtest(df, MACDOptStrategy, cash=cash, commission=commission)
        result = bt.run()
        
        results.append({
            'fast': fast,
            'slow': slow,
            'signal': signal,
            'Return [%]': result['Return [%]'],
            'Sharpe Ratio': result['Sharpe Ratio'],
            'Max. Drawdown [%]': result['Max. Drawdown [%]'],
            'Win Rate [%]': result['Win Rate [%]'],
            '# Trades': result['# Trades'],
        })
    
    return pd.DataFrame(results)

# グリッド検索パラメータ
FAST_RANGE = [10, 12, 14]
SLOW_RANGE = [24, 26, 28]
SIGNAL_RANGE = [7, 9, 11]

print("\nMACD グリッド検索を実行中...")
print(f"Fast: {FAST_RANGE}, Slow: {SLOW_RANGE}, Signal: {SIGNAL_RANGE}")

macd_results = grid_search_macd(sample_df, FAST_RANGE, SLOW_RANGE, SIGNAL_RANGE)

print(f"\n✓ グリッド検索完了: {len(macd_results)} 組合せを試行")
print(f"\nTop 5 結果（Sharpe 比率で評価）:")
print(macd_results.nlargest(5, 'Sharpe Ratio')[['fast', 'slow', 'signal', 'Return [%]', 'Sharpe Ratio', '# Trades']])


MACD グリッド検索を実行中...
Fast: [10, 12, 14], Slow: [24, 26, 28], Signal: [7, 9, 11]


Backtest.run:   0%|          | 0/472 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/470 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/468 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/470 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/466 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/468 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/466 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/464 [00:00<?, ?bar/s]


✓ グリッド検索完了: 27 組合せを試行

Top 5 結果（Sharpe 比率で評価）:
    fast  slow  signal  Return [%]  Sharpe Ratio  # Trades
26    14    28      11   -6.391928     -0.280592        18
23    14    26      11   -8.874630     -0.388904        19
20    14    24      11   -9.303826     -0.416922        18
14    12    26      11  -10.512222     -0.475973        18
22    14    26       9  -12.039948     -0.552286        19


## 6. RSI パラメータ最適化

### このセルの説明
- 何をしているか？
  - 短期 RSI と長期 RSI の期間を変えながらグリッド探索し、バックテスト成績を収集しています。
- 何のためにしているか？
  - `ta.RSI` の期間引数を最適化し、過剰売買を抑えつつ有効なシグナル設定を見つけるためです。
- 理論的背景とモデル
  - RSI は
  $$
  RSI_t = 100-\frac{100}{1+RS_t},\quad RS_t=\frac{\text{AvgGain}_t}{\text{AvgLoss}_t}
  $$
  で定義されます。
  - 本セルでは $\theta=(n_1,n_2)$ の 2 変数最適化として、$n_1<n_2$ 制約の下で性能を比較します。

In [36]:
def grid_search_rsi(df: pd.DataFrame,
                    short_rsi_range: list,
                    long_rsi_range: list,
                    cash: float = CASH,
                    commission: float = COMMISSION) -> pd.DataFrame:
    """
    RSI クロス戦略のグリッド検索
    """
    results = []
    
    for n1, n2 in itertools.product(short_rsi_range, long_rsi_range):
        if n1 >= n2:
            continue
        
        class RSIOptStrategy(RSICrossStrategy):
            pass
        
        RSIOptStrategy.n1 = n1
        RSIOptStrategy.n2 = n2
        
        bt = Backtest(df, RSIOptStrategy, cash=cash, commission=commission)
        result = bt.run()
        
        results.append({
            'n1': n1,
            'n2': n2,
            'Return [%]': result['Return [%]'],
            'Sharpe Ratio': result['Sharpe Ratio'],
            'Max. Drawdown [%]': result['Max. Drawdown [%]'],
            'Win Rate [%]': result['Win Rate [%]'],
            '# Trades': result['# Trades'],
        })
    
    return pd.DataFrame(results)

# グリッド検索パラメータ
SHORT_RSI_RANGE = [10, 14, 21]
LONG_RSI_RANGE = [28, 35, 42]

print("\nRSI グリッド検索を実行中...")
rsi_results = grid_search_rsi(sample_df, SHORT_RSI_RANGE, LONG_RSI_RANGE)

print(f"\n✓ グリッド検索完了: {len(rsi_results)} 組合せを試行")
print(f"\nTop 5 結果（Sharpe 比率で評価）:")
print(rsi_results.nlargest(5, 'Sharpe Ratio')[['n1', 'n2', 'Return [%]', 'Sharpe Ratio', '# Trades']])


RSI グリッド検索を実行中...


Backtest.run:   0%|          | 0/473 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/466 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/459 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/473 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/459 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/473 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/466 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/459 [00:00<?, ?bar/s]


✓ グリッド検索完了: 9 組合せを試行

Top 5 結果（Sharpe 比率で評価）:
   n1  n2  Return [%]  Sharpe Ratio  # Trades
8  21  42   -6.198737     -0.262485        30
6  21  28  -12.959595     -0.568142        33
2  10  42  -14.375326     -0.648278        43
5  14  42  -15.403713     -0.682092        37
7  21  35  -17.141881     -0.774616        32


## 7. 複数銘柄での汎化性能検証

最適パラメータが他の銘柄でも有効か確認します。

### このセルの説明
- 何をしているか？
  - 単一銘柄で得た最適パラメータを複数銘柄へ適用し、ローカル通貨リターンと為替考慮後の円建て損益を同時に評価しています。
- 何のためにしているか？
  - 売買時点の為替レート差を損益へ反映し、実際の円建て投資に近い成績を確認するためです。
- 理論的背景とモデル
  - 買付日 $t_b$ の対象通貨建て買付金額を $B_{ccy}$、為替レート（JPY/ccy）を $FX_{t_b}$ とすると、円建て買付金額は
  $$
  B_{JPY}=B_{ccy}\times FX_{t_b}
  $$
  - 売却日 $t_s$ の対象通貨建て売却金額を $S_{ccy}$、為替レートを $FX_{t_s}$ とすると、円建て売却金額は
  $$
  S_{JPY}=S_{ccy}\times FX_{t_s}
  $$
  - 円建て損益と円建て収益率は
  $$
  PnL_{JPY}=S_{JPY}-B_{JPY},\quad Return_{JPY}=\frac{PnL_{JPY}}{B_{JPY}}\times100
  $$
  で計算します。

In [37]:
# 各戦略の最適パラメータを抽出
best_sma = sma_results.loc[sma_results['Sharpe Ratio'].idxmax()]
best_macd = macd_results.loc[macd_results['Sharpe Ratio'].idxmax()]
best_rsi = rsi_results.loc[rsi_results['Sharpe Ratio'].idxmax()]

print("\n" + "=" * 60)
print(f"最適パラメータ（{TARGET_STOCKS[0]['name']}の場合）")
print("=" * 60)
print(f"\nSMA: n1={int(best_sma['n1'])}, n2={int(best_sma['n2'])}")
print(f"  Return: {best_sma['Return [%]']:.2f}%, Sharpe: {best_sma['Sharpe Ratio']:.2f}")
print(f"\nMACD: fast={int(best_macd['fast'])}, slow={int(best_macd['slow'])}, signal={int(best_macd['signal'])}")
print(f"  Return: {best_macd['Return [%]']:.2f}%, Sharpe: {best_macd['Sharpe Ratio']:.2f}")
print(f"\nRSI: n1={int(best_rsi['n1'])}, n2={int(best_rsi['n2'])}")
print(f"  Return: {best_rsi['Return [%]']:.2f}%, Sharpe: {best_rsi['Sharpe Ratio']:.2f}")

print("\n" + "=" * 60)
print("複数銘柄での最適パラメータ検証（為替考慮）")
print("=" * 60)

multi_stock_results = []

for stock_info in TARGET_STOCKS:
    code = stock_info["code"]
    market = stock_info["market"]
    name = stock_info["name"]
    currency = stock_info.get("currency", "JPY")

    try:
        df_test = fetch_stock_data(code, market, DATA_START, DATA_END)
    except Exception as e:
        print(f"\n✗ {name}: データ取得失敗 - {e}")
        continue

    class SMA_BestStrategy(SMACrossStrategy):
        pass
    SMA_BestStrategy.n1 = int(best_sma['n1'])
    SMA_BestStrategy.n2 = int(best_sma['n2'])

    bt_sma = Backtest(df_test, SMA_BestStrategy, cash=CASH, commission=COMMISSION)
    result_sma = bt_sma.run()
    sma_jpy = calc_jpy_trade_amounts(result_sma, currency)

    class MACD_BestStrategy(MACDCrossStrategy):
        pass
    MACD_BestStrategy.fast = int(best_macd['fast'])
    MACD_BestStrategy.slow = int(best_macd['slow'])
    MACD_BestStrategy.signal_period = int(best_macd['signal'])

    bt_macd = Backtest(df_test, MACD_BestStrategy, cash=CASH, commission=COMMISSION)
    result_macd = bt_macd.run()
    macd_jpy = calc_jpy_trade_amounts(result_macd, currency)

    class RSI_BestStrategy(RSICrossStrategy):
        pass
    RSI_BestStrategy.n1 = int(best_rsi['n1'])
    RSI_BestStrategy.n2 = int(best_rsi['n2'])

    bt_rsi = Backtest(df_test, RSI_BestStrategy, cash=CASH, commission=COMMISSION)
    result_rsi = bt_rsi.run()
    rsi_jpy = calc_jpy_trade_amounts(result_rsi, currency)

    multi_stock_results.append({
        '銘柄': name,
        '銘柄コード': code,
        '通貨': currency,
        'SMA_Return': result_sma['Return [%]'],
        'SMA_Sharpe': result_sma['Sharpe Ratio'],
        'SMA_PnL_JPY': sma_jpy['pnl_jpy'],
        'SMA_Return_JPY[%]': sma_jpy['return_jpy_pct'],
        'MACD_Return': result_macd['Return [%]'],
        'MACD_Sharpe': result_macd['Sharpe Ratio'],
        'MACD_PnL_JPY': macd_jpy['pnl_jpy'],
        'MACD_Return_JPY[%]': macd_jpy['return_jpy_pct'],
        'RSI_Return': result_rsi['Return [%]'],
        'RSI_Sharpe': result_rsi['Sharpe Ratio'],
        'RSI_PnL_JPY': rsi_jpy['pnl_jpy'],
        'RSI_Return_JPY[%]': rsi_jpy['return_jpy_pct'],
    })

    print(f"\n{name} ({code}) [{currency}]:")
    print(f"  SMA  : Local Return {result_sma['Return [%]']:>7.2f}% | JPY PnL {sma_jpy['pnl_jpy']:>12,.0f} | JPY Return {sma_jpy['return_jpy_pct']:>7.2f}%")
    print(f"  MACD : Local Return {result_macd['Return [%]']:>7.2f}% | JPY PnL {macd_jpy['pnl_jpy']:>12,.0f} | JPY Return {macd_jpy['return_jpy_pct']:>7.2f}%")
    print(f"  RSI  : Local Return {result_rsi['Return [%]']:>7.2f}% | JPY PnL {rsi_jpy['pnl_jpy']:>12,.0f} | JPY Return {rsi_jpy['return_jpy_pct']:>7.2f}%")

multi_df = pd.DataFrame(multi_stock_results)
print(f"\n✓ 複数銘柄検証完了: {len(multi_df)} 銘柄")


最適パラメータ（GDの場合）

SMA: n1=7, n2=50
  Return: 16.71%, Sharpe: 0.54

MACD: fast=14, slow=28, signal=11
  Return: -6.39%, Sharpe: -0.28

RSI: n1=21, n2=42
  Return: -6.20%, Sharpe: -0.26

複数銘柄での最適パラメータ検証（為替考慮）


Backtest.run:   0%|          | 0/452 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/464 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/459 [00:00<?, ?bar/s]


GD (GD) [USD]:
  SMA  : Local Return   16.71% | JPY PnL    6,739,617 | JPY Return    5.97%
  MACD : Local Return   -6.39% | JPY PnL      198,862 | JPY Return    0.07%
  RSI  : Local Return   -6.20% | JPY PnL       13,882 | JPY Return    0.00%


...


GDXJ (GDXJ) [USD]:
  SMA  : Local Return  -16.78% | JPY PnL   -1,188,054 | JPY Return   -1.05%
  MACD : Local Return  -13.28% | JPY PnL      -48,813 | JPY Return   -0.02%
  RSI  : Local Return   16.26% | JPY PnL    4,607,888 | JPY Return    1.52%


Backtest.run:   0%|          | 0/143 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/155 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/150 [00:00<?, ?bar/s]


GEV (GEV) [USD]:
  SMA  : Local Return   75.21% | JPY PnL            0 | JPY Return     nan%
  MACD : Local Return   52.45% | JPY PnL    8,072,949 | JPY Return    6.98%
  RSI  : Local Return   62.48% | JPY PnL   10,489,735 | JPY Return   18.99%

✓ 複数銘柄検証完了: 4 銘柄


## 8. 結果の可視化

グリッド検索結果をヒートマップで可視化します。

### このセルの説明
- 何をしているか？
  - ヒートマップ、棒グラフ、散布図で最適化結果を可視化し、パラメータ地形を分析しています。
- 何のためにしているか？
  - 数値表だけでは見えない感度やトレードオフ（収益・リスク・取引回数）を把握するためです。
- 理論的背景とモデル
  - ヒートマップは評価関数 $J(\theta)$ の離散近似面を示します。
  $$
  J: \Theta \to \mathbb{R},\quad \theta=(n_1,n_2)
  $$
  - 散布図では、各試行点を
  $$
  (x,y,s,c)=(\#Trades,\ Return,\ |MDD|,\ Sharpe)
  $$
  として多次元情報を同時に表現します。

In [38]:
# SMA ヒートマップ（Sharpe 比率）
sma_heatmap = sma_results.pivot_table(
    values='Sharpe Ratio',
    index='n2',
    columns='n1',
    aggfunc='first'
)

fig_sma = go.Figure(
    data=go.Heatmap(
        z=sma_heatmap.values,
        x=sma_heatmap.columns,
        y=sma_heatmap.index,
        colorscale='RdYlGn',
        text=np.round(sma_heatmap.values, 2),
        texttemplate='%{text:.2f}',
        textfont={'size': 12},
        hovertemplate='短期 MA: %{x}<br>長期 MA: %{y}<br>Sharpe: %{z:.2f}<extra></extra>'
    )
)
fig_sma.update_layout(
    title='SMA グリッド検索: Sharpe 比率',
    xaxis_title='短期 MA 日数 (n1)',
    yaxis_title='長期 MA 日数 (n2)',
    height=400,
    width=600
)
fig_sma.show()

# 複数銘柄結果の比較
fig_multi = px.bar(
    multi_df,
    x='銘柄',
    y=['SMA_Sharpe', 'MACD_Sharpe', 'RSI_Sharpe'],
    barmode='group',
    title='複数銘柄での最適パラメータ性能（Sharpe 比率）',
    labels={'value': 'Sharpe 比率', 'variable': '戦略'},
    height=400
)
fig_multi.show()

# SMA リターン分布
sma_scatter_df = sma_results.copy()
sma_scatter_df['Drawdown Magnitude [%]'] = sma_scatter_df['Max. Drawdown [%]'].abs()

fig_returns = px.scatter(
    sma_scatter_df,
    x='# Trades',
    y='Return [%]',
    color='Sharpe Ratio',
    size='Drawdown Magnitude [%]',
    hover_data=['n1', 'n2', 'Max. Drawdown [%]'],
    title='SMA グリッド検索: リターン vs トレード数',
    labels={
        '# Trades': 'トレード数',
        'Return [%]': 'リターン (%)',
        'Drawdown Magnitude [%]': '最大ドローダウンの絶対値 (%)',
    },
    color_continuous_scale='RdYlGn'
)
fig_returns.show()

print('✓ 可視化完了')

✓ 可視化完了


## 9. 結果の保存

グリッド検索結果と最適パラメータを保存します。

### このセルの説明
- 何をしているか？
  - 最適化で得た最良パラメータと、全試行結果を JSON/CSV として保存しています。
- 何のためにしているか？
  - 実験結果の再現性を担保し、後続の比較実験や監査に利用できる形で永続化するためです。
- 理論的背景とモデル
  - 最適化の解を
  $$
  \theta^*=\arg\max_{\theta\in\Theta}J(\theta)
  $$
  とすると、保存対象は $\theta^*$ とその評価値 $J(\theta^*)$、および全試行集合
  $$
  \mathcal{R}=\{(\theta_i,J(\theta_i))\}_{i=1}^{K}
  $$
  です。これにより追試可能性が確保されます。

### このセルの説明
- 何をしているか？
  - 最適パラメータと全試行ログを JSON/CSV で保存し、再利用可能な成果物を出力しています。
- 何のためにしているか？
  - 研究結果を永続化し、再現実験・比較実験・レポーティングを可能にするためです。
- 理論的背景とモデル
  - 最適化問題の解を
  $$
  \theta^*=\arg\max_{\theta\in\Theta} J(\theta)
  $$
  としたとき、保存対象は $(\theta^*, J(\theta^*))$ と試行集合
  $$
  \{(\theta_i, J(\theta_i))\}_{i=1}^{K}
  $$
  です。これにより追試と監査可能性を担保できます。

In [39]:
import json
from datetime import datetime

results_dir = Path('/workspace/results')
results_dir.mkdir(parents=True, exist_ok=True)

optimal_params = {
    'timestamp': datetime.now().isoformat(),
    'stock': {
        'code': TARGET_STOCKS[0]['code'],
        'name': TARGET_STOCKS[0]['name'],
        'market': TARGET_STOCKS[0]['market']
    },
    'test_period': {
        'start': DATA_START.isoformat(),
        'end': DATA_END.isoformat()
    },
    'sma': {
        'n1': int(best_sma['n1']),
        'n2': int(best_sma['n2']),
        'return_pct': float(best_sma['Return [%]']),
        'sharpe': float(best_sma['Sharpe Ratio']),
        'max_drawdown_pct': float(best_sma['Max. Drawdown [%]']),
    },
    'macd': {
        'fast': int(best_macd['fast']),
        'slow': int(best_macd['slow']),
        'signal': int(best_macd['signal']),
        'return_pct': float(best_macd['Return [%]']),
        'sharpe': float(best_macd['Sharpe Ratio']),
    },
    'rsi': {
        'n1': int(best_rsi['n1']),
        'n2': int(best_rsi['n2']),
        'return_pct': float(best_rsi['Return [%]']),
        'sharpe': float(best_rsi['Sharpe Ratio']),
    }
}

sma_results.to_csv(results_dir / 'sma_grid_search.csv', index=False)
macd_results.to_csv(results_dir / 'macd_grid_search.csv', index=False)
rsi_results.to_csv(results_dir / 'rsi_grid_search.csv', index=False)
multi_df.to_csv(results_dir / 'multi_stock_validation.csv', index=False)

with open(results_dir / 'optimal_parameters.json', 'w') as f:
    json.dump(optimal_params, f, indent=2, ensure_ascii=False)

print("✓ 結果を保存しました:")
print(f"  - {results_dir / 'sma_grid_search.csv'}")
print(f"  - {results_dir / 'macd_grid_search.csv'}")
print(f"  - {results_dir / 'rsi_grid_search.csv'}")
print(f"  - {results_dir / 'multi_stock_validation.csv'}")
print(f"  - {results_dir / 'optimal_parameters.json'}")

print("\n" + "=" * 60)
print("リサーチサマリー")
print("=" * 60)
print(f"\n実施期間: {DATA_START.date()} ～ {DATA_END.date()}")
print(f"対象銘柄: {TARGET_STOCKS[0]['name']}")
print(f"グリッド検索試行数: SMA={len(sma_results)}, MACD={len(macd_results)}, RSI={len(rsi_results)}")
print("\n最適パラメータ:")
print(f"  SMA: ({int(best_sma['n1'])}, {int(best_sma['n2'])}) → Sharpe {best_sma['Sharpe Ratio']:.2f}")
print(f"  MACD: ({int(best_macd['fast'])}, {int(best_macd['slow'])}, {int(best_macd['signal'])}) → Sharpe {best_macd['Sharpe Ratio']:.2f}")
print(f"  RSI: ({int(best_rsi['n1'])}, {int(best_rsi['n2'])}) → Sharpe {best_rsi['Sharpe Ratio']:.2f}")

✓ 結果を保存しました:
  - /workspace/results/sma_grid_search.csv
  - /workspace/results/macd_grid_search.csv
  - /workspace/results/rsi_grid_search.csv
  - /workspace/results/multi_stock_validation.csv
  - /workspace/results/optimal_parameters.json

リサーチサマリー

実施期間: 2023-01-01 ～ 2024-12-31
対象銘柄: GD
グリッド検索試行数: SMA=20, MACD=27, RSI=9

最適パラメータ:
  SMA: (7, 50) → Sharpe 0.54
  MACD: (14, 28, 11) → Sharpe -0.28
  RSI: (21, 42) → Sharpe -0.26


## 10. 次のステップ

### 本ノートブックで実施した内容
✅ 3つの取引ルール（SMA/MACD/RSI）のパラメータをグリッド検索で最適化  
✅ 複数銘柄での汎化性能を検証  
✅ 最適パラメータを JSON/CSV で保存  

### 今後のリサーチ計画
1. **ウォークフォワード分析**（stock_10_walkforward.ipynb）
   - In-sample / Out-of-sample の分割で過学習チェック
   - 3ヶ月ごと / 6ヶ月ごとのローリング検証

2. **リスク管理機能**（stock_11_risk_management.ipynb）
   - ATR ベースの動的ストップロス
   - Kelly 基準によるポジションサイジング
   - ドローダウン管理

3. **ポートフォリオ最適化**（stock_12_portfolio_optimization.ipynb）
   - 複数銘柄での最適配分
   - 相関分析

---

## 参考：Copilot Instructions より

### グリッド検索の形式式
$$\theta^*=\arg\max_{\theta\in\Theta} J(\theta)$$

ここで $\theta$ はパラメータ辞書（例: `{'n1': 5, 'n2': 25}`）、
$J(\theta)$ は評価指標（`Sharpe Ratio` など）です。

### Backtesting.py での最適化
```python
bt = Backtest(data, Strategy, ...)
stats = bt.optimize(...)  # グリッド検索自動実行
```

本ノートブックでは明示的なループ実装で、各試行の詳細ログを記録しています。